## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [8]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [9]:
api_key = os.getenv('OPENROUTER_API_KEY')

In [10]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [13]:
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(
    # temprature=0,
    model="openai/gpt-4o-mini", # OpenRouter ID: openai/gpt-5-nano, anthropic/claude-3.5-sonnet etc.
    api_key=api_key, # sk-or-v1-...
    base_url="https://openrouter.ai/api/v1",
    # optional but recommended by OpenRouter
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "llm_engineering"
    }
)
llm.invoke("hello")
## llm = ChatOpenAI(temperature=0, model_name=MODEL)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 7.2e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 7.2e-06, 'upstream_inference_prompt_cost': 1.2e-06, 'upstream_inference_completions_cost': 6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_369e662417', 'id': 'gen-1789215265-dO8fdcOQloYfjeUEpiHe', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0958a-4f16-77b0-aa88-b7a36e8cf103-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens':

In [14]:
retriever.invoke("Who is Avery?")

[Document(id='49cd89c4-4209-4349-b63b-f306948dfca6', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [15]:
llm.invoke("Who is Avery?")

AIMessage(content='The name "Avery" can refer to various individuals, characters, or concepts depending on the context. It may be a common first name or surname in various cultures. Additionally, “Avery” could be the name of a brand, company, or even a fictional character in a book, movie, or TV show.\n\nIf you have a specific context in mind, such as a particular Avery in entertainment, literature, or a real-life individual, please provide more details, and I can give you a more accurate answer.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 11, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 6.525e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6

In [22]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [23]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [24]:
answer_question("Who is Averi Lancaster?", [])

"Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. Born on March 15, 1985, she co-founded the company in 2015 and has been instrumental in guiding Insurellm to become a leading provider in the Insurance Tech sector. Avery is recognized for her innovative leadership strategies and expertise in risk management, which have played a significant role in the company's successful positioning in the mainstream insurance market. Prior to her role at Insurellm, she served as a Senior Product Manager at Innovate Insurance Solutions, where she developed innovative insurance products geared towards the tech sector. Avery's current salary is $225,000."

In [25]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
